# ML-03 — Frame Your Lane as an ML Task

This notebook frames the Freestyle project as a **binary classification task whose predicted probability becomes a ranking score** for a human content-review queue.

> Research question: Can a compact, public-safe set of search and content signals produce a useful ranking of pages associated with observed performance decline, without using the decline label itself as a feature?


## 1. My lane as an ML task (type)

**Task type: classification, used for ranking/scoring.** Each content item receives a binary observed decline proxy (`trend_direction == "down"`). A classifier estimates the probability associated with that proxy; sorting by that probability creates a review-priority ranking.

This is more useful than a pure rule because the operational output is an ordered queue, not just a yes/no flag.


In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("data/raw/content_refresh_anonymized.csv")
if not DATA.exists():
    DATA = next(Path.cwd().rglob("content_refresh_anonymized.csv"))

df = pd.read_csv(DATA)
print(f"Loaded {len(df):,} content items from the bundled anonymized starter slice.")
print("One row represents one content item/page.")


Loaded 30,000 content items from the bundled anonymized starter slice.
One row represents one content item/page.


## 2. Target or proxy

The target is an **observed snapshot-based proxy**, not a future Google-ranking outcome. I define `decline_target = 1` when `trend_direction` is `down`, and `0` otherwise. The trend/label fields are not allowed to become model features because that would leak the answer into the inputs.

The proxy is useful for this assignment because it gives a measurable outcome for evaluating a prioritization model while keeping the interpretation explicitly non-causal.


In [2]:
df["decline_target"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print("decline_target definition: trend_direction == 'down'")
print(f"decline_target = 1: {df['decline_target'].sum():,} rows")
print(f"decline_target = 0: {(1 - df['decline_target']).sum():,} rows")
print(f"declining proxy rate: {df['decline_target'].mean():.3f}")


decline_target definition: trend_direction == 'down'
decline_target = 1: 16,262 rows
decline_target = 0: 13,738 rows
declining proxy rate: 0.542


## 3. Success metric

**Primary metric: Precision@50.** The practical use case is a limited human review queue, so the key question is how many of the first 50 ranked items match the observed decline proxy.

**Secondary metrics: ROC-AUC and Average Precision.** These measure ranking/discrimination quality across the full evaluation set. Recall and F1 are useful supporting metrics, but they are not the main operational objective.

There is no arbitrary claim that one metric proves the model is useful; the metric is tied to the decision the score supports.


In [3]:
print("Primary metric: Precision@50")
print("Secondary metrics: ROC-AUC and Average Precision")
print("Operational action: prioritize a small human review queue.")


Primary metric: Precision@50
Secondary metrics: ROC-AUC and Average Precision
Operational action: prioritize a small human review queue.


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one row = one content item/page.** The dataframe below uses only public-safe analytical fields; pseudonymous client identifiers are intentionally not displayed.


In [4]:
cols = [
    "content_id", "word_count", "char_count", "impressions_90d", "clicks_90d",
    "sessions_90d", "content_age_days", "avg_position", "trend_direction", "decline_target"
]
display(df[cols].head(5))
print("rows in dataframe:", len(df))
print("unique content_id:", df["content_id"].nunique())


        content_id  word_count  char_count  impressions_90d  clicks_90d  sessions_90d  content_age_days  avg_position trend_direction  decline_target0  content_304f48230142        3221       20457              3803          29            17               187          10.6            down              11  content_a1fb4e703a9e        2481       15562             15320           7             9               445          20.3            down              12  content_9aa793d4d895        3515       23643             12581          11            11               141          36.5            down              13  content_331d6c4de07b        0           0             11751          58            78               463           6.2          stable              04  content_d99b7a2d90ca        2803       17469             19140          24           145               263          44.0            down              1

### Feature boundary

Candidate predictors are signals available in the starter slice, such as search demand, impressions/clicks, position, engagement, content age, content length, and other non-label fields. I exclude `trend_direction`, `trend_pct`, IDs, and any fields that directly encode the outcome. This keeps the framing aligned with a leakage-aware ranking problem.


In [5]:
excluded = ["trend_direction", "trend_pct", "content_id", "client_id"]
print("Excluded from model features:", ", ".join(excluded))
print("Candidate inputs include: impressions/clicks, position, engagement, content age, content length, search demand and other non-label fields.")


Excluded from model features: trend_direction, trend_pct, content_id, client_id
Candidate inputs include: impressions/clicks, position, engagement, content age, content length, search demand and other non-label fields.


## 5. Why ML beats a fixed rule here

A fixed rule such as `avg_position > X` or `impressions_last_30d < X` uses one threshold and can miss interactions between signals. The observed dataset contains content items with very different traffic volumes, ages, positions, engagement levels, and content lengths. A model can combine these signals into a continuous score and therefore produce a more flexible priority order.

The important distinction is not that ML is automatically better. It is that ML gives us a testable way to compare a multi-signal score against a transparent fixed-rule baseline on the same validation split.


In [6]:
print(f"The starter slice has {len(df):,} content items with heterogeneous traffic, position, age, and content-length signals.")
print("A fixed threshold can rank or flag only from the chosen rule; a model can combine multiple signals and output a continuous priority score.")
print("The claim of improvement must be validated against the fixed-rule baseline on the same split.")


The starter slice has 30,000 content items with heterogeneous traffic, position, age, and content-length signals.
A fixed threshold can rank or flag only from the chosen rule; a model can combine multiple signals and output a continuous priority score.
The claim of improvement must be validated against the fixed-rule baseline on the same split.


## 6. Self-check

- [x] ML task type is named: classification, with probability used for ranking/scoring.
- [x] Target/proxy is defined: `trend_direction == "down"`.
- [x] Success metrics are tied to the action: Precision@50 primary; ROC-AUC and Average Precision secondary.
- [x] Unit of analysis is shown as a real dataframe: one row per content item/page.
- [x] Feature leakage is addressed: outcome/label fields are excluded from candidate inputs.
- [x] The output supports a real action: a human content-refresh review queue.
- [x] Public-safe framing is used: observed/associated/decision-support, not causal claims.


### Final framing

This notebook turns the Freestyle idea into a measurable ML problem: **predict an observed decline proxy for each content item, then use the predicted probability as a ranking score for a limited human review queue.** The next stages can compare a transparent baseline with ML models using the same validation design and leakage controls.